<center><img src='https://drive.google.com/uc?id=1_utx_ZGclmCwNttSe40kYA6VHzNocdET' height="60"></center>

AI TECH - Akademia Innowacyjnych Zastosowań Technologii Cyfrowych. Program Operacyjny Polska Cyfrowa na lata 2014-2020
<hr>

<center><img src='https://drive.google.com/uc?id=1BXZ0u3562N_MqCLcekI-Ens77Kk4LpPm'></center>

<center>
Projekt współfinansowany ze środków Unii Europejskiej w ramach Europejskiego Funduszu Rozwoju Regionalnego
Program Operacyjny Polska Cyfrowa na lata 2014-2020,
Oś Priorytetowa nr 3 "Cyfrowe kompetencje społeczeństwa" Działanie  nr 3.2 "Innowacyjne rozwiązania na rzecz aktywizacji cyfrowej"
Tytuł projektu:  „Akademia Innowacyjnych Zastosowań Technologii Cyfrowych (AI Tech)”
    </center>

# MSLE and gradients

Today we're implementing Gradient Descent by hand.
We'll also look at the Mean Squared Logarithmic Error (MSLE).

In [ ]:
%matplotlib inline

from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from tqdm.auto import tqdm

NDArray = np.ndarray[Any, Any]

np.set_printoptions(precision=4, suppress=True)
np.random.seed(357)

## Loading and converting data

Same data as before, cleaned up. We'll use NumPy arrays instead of Pandas DataFrames to simplify the implementation.

In [ ]:
!wget --no-verbose -O lab1_apartments_train_clean.csv https://mimuw.edu.pl/~mwrochna/upload/lab1_apartments_train_clean.csv
!wget --no-verbose -O lab1_apartments_test_clean.csv https://mimuw.edu.pl/~mwrochna/upload/lab1_apartments_test_clean.csv
!head lab1_apartments_train_clean.csv lab1_apartments_test_clean.csv

In [ ]:
def load_dataset(path: str) -> tuple[NDArray, NDArray]:
    """
    Returns (x, y) where:
    - x: input features, shape (n_apartments, n_features)
    - y: prices, shape (n_apartments,)
    """
    data = pd.read_csv(path)
    y = data["price"].to_numpy()
    x = data.loc[:, data.columns != "price"].to_numpy()
    return x, y

x_train, y_train = load_dataset("lab1_apartments_train_clean.csv")
x_test, y_test = load_dataset("lab1_apartments_test_clean.csv")

print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

In [ ]:
# Convert categorical features to numbers (one ordinal column, for simplicity).
ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

ordinal_encoder.fit(x_train[:, [1]])

x_train[:, [1]] = ordinal_encoder.transform(x_train[:, [1]])
x_test[:, [1]] = ordinal_encoder.transform(x_test[:, [1]])

# Convert ints to float.
x_train = x_train.astype(np.float64)
x_test = x_test.astype(np.float64)

# Preview the first three rows.
x_train[:3]

## The loss, constant models

The range of prices is quite large.

In [ ]:
print(np.min(y_train), np.max(y_train), np.mean(y_train))

When predicting them, relative errors are arguably more important than absolute errors.<br>
That is, mistaking 1M vs 2M is arguably similar to mistaking 11M vs 22M, more so than mistaking 11M vs 12M.

Therefore, let's assume that our predictions should minimize the so-called *mean squared logarithmic error*:
$$
MSLE(\bar{y}, \bar{p}) = \frac{1}{n} \sum_{i=1}^n (\log(1+y_i) - \log(1+p_i))^2,
$$
where $y_i$ is the ground truth (the target), and $p_i$ is our prediction.

Let's implement the loss function first.

In [ ]:
def mse(ys: NDArray, ps: NDArray) -> np.float64:
    assert ys.shape == ps.shape
    return np.mean((ys - ps) * (ys - ps))

In [ ]:
def msle(ys: NDArray, ps: NDArray) -> np.float64:
    assert ys.shape == ps.shape
    #################################################
    # TODO: Implement this function (without loops) #
    #################################################


The simplest model is predicting the same constant for each instance. Test your implementation of msle against outputing the mean price.

In [ ]:
###################################################
# TODO: Compute msle for outputing the mean price #
###################################################


To get a feel of whether this error is high or low, we can apply exp and sqrt to get some kind of mean (`exp of quadratic mean of log`) in interpretable units.<br>
(For context, observe that `exp of arithmetic mean of log` is the geometric mean).

So we are roughly 1.92× off the actual y_test when we predict that constant value all the time, on average.

Recall that outputing the **mean** minimizes $MSE$. However, we're now dealing with $MSLE$.

Find a constant that should result in the lowest $MSLE$.

In [ ]:
#############################################
# TODO: Find this constant and compute msle #
#############################################

## Linear regression (standard)

Now, let's implement training of a standard linear regression model via gradient descent.

In [ ]:
def train(
    x: NDArray, y: NDArray, alpha: float = 1e-7, n_iterations: int = 100000
) -> tuple[NDArray, np.float64]:
    """Linear regression (which optimizes MSE). Returns (weights, bias)."""

    # B is batch size (number of observations).
    # D is number of (input) features.
    B, D = x.shape
    assert y.shape == (B,)

    ### TODO ###

    return weights, bias

weights, bias = train(x_train, y_train)
preds_test = x_test @ weights + bias  # shapes (B, D) @ (D,) + ()
print("test MSLE:", msle(y_test, preds_test))
# Should be much better than the constant model.

## Linear regression (MSLE)

Note that the loss function that the algorithms optimizes (i.e $MSE$) differs from $MSLE$. We've already seen that this may result in a suboptimal solution.

How can you change the setting so that we optimze $MSLE$ instead?

Hint:
<sub><sup><sub><sup><sub><sup>
Be lazy. We don't want to change the algorithm.
Use the chain rule and previous computations to get formulas for the gradient.
</sup></sub></sup></sub></sup></sub>

In [ ]:
def train_msle(
    x: NDArray, y: NDArray, alpha: float = 1e+4, n_iterations: int = 50000
) -> tuple[NDArray, NDArray]:
    """Gradient descent for MSLE."""

    #############################################
    # TODO: Optimize msle and compare the error #
    #############################################

    # B is batch size (number of observations).
    # D is number of (input) features.
    B, D = x.shape
    assert y.shape == (B,)


    return weights, bias


weights, bias = train_msle(x_train, y_train)
preds_test = x_test @ weights + bias
print("test MSLE: ", msle(y_test, preds_test))
# Train MSLE should be better. Test MSLE not necessarily.